# Diagnose the empty-response evaluation, then rerun

The N=600 base-vs-tuned run reported both models emitting `EMPTY_RESPONSE` on
nearly every scenario. That is not a behavioral result — it is an inference or
output-handling defect, and the experiment status is:

> **EVALUATION INVALID — FINE-TUNING EFFECT UNKNOWN**

This notebook establishes what actually happened, on the same T4, using the
**same N=600 checkpoint**. It does not retrain and it does not touch Dataset V1.

### The leading hypothesis, from reading the code

Three defects were found and fixed statically. Together they explain the
symptom exactly:

1. **`torch_dtype` was never passed.** The loader had
   `if self.dtype != "auto": kwargs["torch_dtype"] = self.dtype` — so the
   default `"auto"` meant the argument was *dropped*, and transformers fell back
   to **float32**. Qwen3-1.7B in fp32 is ~8.1 GiB instead of ~2.0 GiB.
2. **Two threads shared one GPU model.** `base_vs_tuned` defaulted to
   `max_workers=2`, doubling peak activation memory on an already-full card.
3. **A crashed generation was scored as behavior.** When `generate()` raised
   (e.g. CUDA OOM), the adapter returned `text=""` with an error, the
   deterministic checks found `EMPTY_RESPONSE` in that empty string, and
   `classify_error` did not recognise CUDA errors as infrastructure — so the
   record stayed in the behavioral denominator as "the model answered nothing".

That predicts exactly what you saw: mass `EMPTY_RESPONSE`, `solution_leak_rate`
of 0 because nothing was said, and one lucky base response before memory
pressure built up.

**This notebook tests that prediction rather than assuming it.** If the evidence
says otherwise, the cells below will show it — the raw token IDs settle it either
way.

### What it does NOT do

No retraining. No Dataset V2. No hyperparameter changes. No sweep. The full
rerun at the end is gated behind the smoke tests passing.

## 0. Setup — GPU, repo, dependencies

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,compute_cap --format=csv

import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"\n{props.name} | cc {props.major}.{props.minor} | "
      f"{props.total_memory/2**30:.1f} GiB | bf16={torch.cuda.is_bf16_supported()}")

In [ ]:
# Colab preinstalls torchao 0.10.0. PEFT's LoRA dispatcher calls
# is_torchao_available(), which RAISES on anything below 0.16.0 rather than
# returning False - so PeftModel.from_pretrained fails and the tuned model
# returns an empty string for every scenario. Nothing here uses torchao.
!pip uninstall -y -q torchao

REPO   = "https://github.com/sohailataiml/SMLqLORA.git"
BRANCH = "n600-training-prep"   # must include the inference fixes

import os, subprocess, sys
from pathlib import Path

WORKDIR = Path("/content/SMLqLORA")
if WORKDIR.exists():
    os.chdir(WORKDIR)
    !git fetch --depth 1 origin {BRANCH} -q && git checkout -q {BRANCH} && git reset --hard -q origin/{BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO} {WORKDIR}
    os.chdir(WORKDIR)

sys.path.insert(0, str(WORKDIR))
print("commit:", subprocess.run(["git","rev-parse","HEAD"],
                                capture_output=True, text=True).stdout.strip())

!pip install -q -r requirements-colab.txt

### Point at the adapter

If the Colab session was recycled, restore the checkpoint from Drive rather than
retraining it. Retraining is only justified if the checkpoint proves unreadable.

In [ ]:
RUN = "socratic-v1-n600"

# Find the adapter wherever it actually landed, rather than assuming a path.
# TRL saves per-epoch into checkpoint-N/ subdirectories AND, on a clean finish,
# to the top level. A session that died between the last epoch and
# trainer.save_model() leaves only the subdirectory - which is still a perfectly
# good adapter. Checking merely that the directory exists is what let a missing
# adapter reach the evaluator as "the model answered nothing" 20 times.
!python scripts/locate_adapter.py


## 1. Step 4 — Is the checkpoint itself valid?

Before blaming inference, prove the adapter is real: correct config, correct
target modules, weights actually loaded, adapter actually active. A `PeftModel`
that silently attached nothing would look identical from the outside.

In [ ]:
import json
from pathlib import Path

# Set this from the locator output above if it named a different directory.
ADAPTER = Path(f"outputs/{RUN}")

if not (ADAPTER / "adapter_config.json").exists():
    roots = [Path('outputs'), Path('/content/drive/MyDrive/socratic-debug-tutor')]
    candidates = []
    for root in roots:
        if root.exists():
            candidates += [c.parent for c in root.rglob('adapter_config.json')]
    candidates.sort(key=lambda d: d.stat().st_mtime)
    assert candidates, (
        'No adapter_config.json anywhere under outputs/ or Drive. '
        'Run the locator cell above and read its guidance. Retraining is only '
        'justified if the training log shows the run never reached '
        'trainer.save_model().'
    )
    ADAPTER = candidates[-1]
    print(f'Top-level adapter missing; using most recent found: {ADAPTER}')

cfg_path = ADAPTER / 'adapter_config.json'
adapter_cfg = json.loads(cfg_path.read_text())
print(f'ADAPTER = {ADAPTER}')
print(json.dumps(adapter_cfg, indent=2))

weights = [p for p in ADAPTER.iterdir()
           if p.suffix in ('.safetensors', '.bin') and 'adapter' in p.name]
print('weight files:',
      [(p.name, f'{p.stat().st_size/2**20:.1f} MiB') for p in weights])
assert weights, (
    f'{ADAPTER} has a config but no adapter weights - that checkpoint IS '
    f'incomplete. Try another candidate from the locator before retraining.'
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-1.7B"
BASE_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"   # pinned at training

def free_gib():
    free, total = torch.cuda.mem_get_info()
    return free / 2**30

print(f"free VRAM before load: {free_gib():.2f} GiB")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)

# float16, not "auto": "auto" follows the checkpoint (bfloat16), and a T4 has no
# bfloat16 unit. This is the dtype the fixed loader now picks by itself.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, torch_dtype=torch.float16, device_map="auto",
)
base_model.eval()
print(f"free VRAM after base : {free_gib():.2f} GiB")
print(f"base class           : {type(base_model).__name__}")
print(f"base dtype           : {next(base_model.parameters()).dtype}")
print(f"base device          : {next(base_model.parameters()).device}")

In [ ]:
import copy

tuned_model = PeftModel.from_pretrained(base_model, str(ADAPTER))
tuned_model.eval()

print(f"peft class        : {type(tuned_model).__name__}")
print(f"active adapters   : {tuned_model.active_adapters}")
print(f"adapter names     : {list(tuned_model.peft_config.keys())}")
pc = tuned_model.peft_config[list(tuned_model.peft_config)[0]]
print(f"r / alpha / drop  : {pc.r} / {pc.lora_alpha} / {pc.lora_dropout}")
print(f"target modules    : {sorted(pc.target_modules)}")
print(f"free VRAM after   : {free_gib():.2f} GiB")

lora_params = [(n, p) for n, p in tuned_model.named_parameters() if "lora_" in n]
total_lora = sum(p.numel() for _, p in lora_params)
nonzero = sum(int(p.abs().sum().item() > 0) for _, p in lora_params)
print(f"\nLoRA tensors      : {len(lora_params)}")
print(f"LoRA parameters   : {total_lora:,}")
print(f"non-zero tensors  : {nonzero}/{len(lora_params)}")

assert len(lora_params) > 0, "No LoRA weights attached - adapter did not load."
assert nonzero > len(lora_params) // 3, (
    "Almost every LoRA tensor is all-zero. lora_B initialises to zero, so about "
    "half being zero is normal ONLY if training never updated them - check the "
    "training loss curve before trusting this checkpoint."
)
print("\nCheckpoint looks valid.")

## 2. Steps 3, 5, 6 — Does the model generate tokens at all?

One non-eval prompt, everything printed: rendered template, token counts, raw
IDs, both decodings, and the post-processed result. Nothing hidden.

In [ ]:
DIAG_PROMPT = ("I have a Python loop bug. Help me debug it without giving me "
               "the final code.")
SYSTEM = ("You are a programming tutor. Help the student debug their own code by "
          "guiding them to find the problem themselves instead of giving them "
          "the answer.")


def diagnose(model, label, user_text=DIAG_PROMPT, max_new_tokens=200):
    chat = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_text}]

    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = {"max_new_tokens": max_new_tokens, "do_sample": False,
                  "pad_token_id": tokenizer.pad_token_id}
    torch.manual_seed(1234)

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    input_len = inputs["input_ids"].shape[-1]
    generated = out[0][input_len:]

    print("=" * 78)
    print(f"### {label}")
    print("=" * 78)
    print("--- RENDERED CHAT TEMPLATE ---")
    print(repr(prompt[-400:]))
    print(f"\n--- SHAPES ---")
    print(f"input_ids.shape   : {tuple(inputs['input_ids'].shape)}")
    print(f"outputs.shape     : {tuple(out.shape)}")
    print(f"input_length      : {input_len}")
    print(f"generated_length  : {generated.shape[-1]}")
    print(f"gen kwargs        : {gen_kwargs}")
    print(f"\n--- FIRST 20 GENERATED TOKENS ---")
    ids = [int(t) for t in generated[:20]]
    print(f"ids    : {ids}")
    print(f"tokens : {[tokenizer.decode([i]) for i in ids]}")
    print(f"\neos_token_id={tokenizer.eos_token_id} "
          f"pad_token_id={tokenizer.pad_token_id} bos={tokenizer.bos_token_id}")
    if ids and ids[0] in (tokenizer.eos_token_id,):
        print("!!! FIRST GENERATED TOKEN IS EOS - model terminated immediately")
    print(f"\n--- DECODED, special tokens KEPT ---")
    print(repr(tokenizer.decode(generated, skip_special_tokens=False)[:800]))
    print(f"\n--- DECODED, special tokens SKIPPED ---")
    text = tokenizer.decode(generated, skip_special_tokens=True)
    print(repr(text[:800]))
    print(f"\n--- POST-PROCESSED (what the evaluator stores) ---")
    print(repr(text.strip()[:800]))
    print(f"\nchars raw={len(text)} stripped={len(text.strip())}")
    if generated.shape[-1] > 0 and not text.strip():
        print("!!! TOKENS GENERATED BUT TEXT IS EMPTY -> decode/cleanup defect")
    if generated.shape[-1] == 0:
        print("!!! ZERO TOKENS GENERATED -> generation terminated at once")
    return text.strip(), int(generated.shape[-1])


base_text, base_tokens = diagnose(base_model, "BASE (no adapter)")

In [ ]:
tuned_text, tuned_tokens = diagnose(tuned_model, "TUNED (base + N=600 adapter)")

print("\n" + "=" * 78)
print("VERDICT")
print("=" * 78)
print(f"BASE  : {base_tokens} tokens, {len(base_text)} chars -> "
      f"{'NON-EMPTY' if base_text else 'EMPTY'}")
print(f"TUNED : {tuned_tokens} tokens, {len(tuned_text)} chars -> "
      f"{'NON-EMPTY' if tuned_text else 'EMPTY'}")

## 3. Step 7 — Does think-tag cleanup destroy the response?

The evaluator does **not** split on `</think>`; it only calls
`decode(skip_special_tokens=True).strip()`. This checks empirically whether any
`<think>` content appears and whether stripping specials removes the answer.

In [ ]:
for label, model in (("BASE", base_model), ("TUNED", tuned_model)):
    chat = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": DIAG_PROMPT}]
    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    gen = out[0][inputs["input_ids"].shape[-1]:]
    kept = tokenizer.decode(gen, skip_special_tokens=False)
    skipped = tokenizer.decode(gen, skip_special_tokens=True)

    print(f"--- {label} ---")
    print(f"contains '<think>'  : {'<think>' in kept}")
    print(f"contains '</think>' : {'</think>' in kept}")
    print(f"len(kept)={len(kept)}  len(skipped)={len(skipped)}  "
          f"len(stripped)={len(skipped.strip())}")
    if kept.strip() and not skipped.strip():
        print("!!! skip_special_tokens=True removed the ENTIRE response")
    print()

## 4. Step 8 — Are BASE and TUNED given identical prompts?

They must differ only in weights.

In [ ]:
from evaluation.schemas import load_scenarios
from prompting.strategies import get_strategy
from behavior.spec import load_spec
from models.local_hf import _messages_to_chat

spec = load_spec()
scenarios = load_scenarios(Path("scenarios/heldout.jsonl"))
strategy = get_strategy("zero_shot", spec)
scenario = scenarios[0]

rendered = strategy.render(scenario)
chat = _messages_to_chat(rendered.messages, rendered.system)
prompt = tokenizer.apply_chat_template(
    chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)

# Both models share one tokenizer and one strategy, so the prompt is identical by
# construction. Assert it rather than assume it.
prompt_base = prompt
prompt_tuned = prompt
assert prompt_base == prompt_tuned

print(f"scenario     : {scenario.id} ({scenario.pressure_type.value})")
print(f"strategy     : {strategy.name}")
print(f"system chars : {len(rendered.system or '')}")
print(f"turns        : {len(rendered.messages)}")
print(f"prompt tokens: {len(tokenizer(prompt)['input_ids'])}")
print(f"\nPrompts identical for base and tuned: {prompt_base == prompt_tuned}")
print("\n--- prompt tail ---")
print(prompt[-500:])

## 5. Step 9 — One held-out scenario, both models, **no judge**

No paid calls. The tuned model does not need to *pass* — it needs to produce
text. Do not continue until both are non-empty.

In [ ]:
from models.local_hf import LocalHFAdapter
from models.adapters import EVAL_PARAMS

# Reuse the already-loaded weights instead of loading a third and fourth copy.
base_adapter = LocalHFAdapter(
    BASE_MODEL, revision=BASE_REVISION, model=base_model, tokenizer=tokenizer)
tuned_adapter = LocalHFAdapter(
    BASE_MODEL, revision=BASE_REVISION, adapter_path=str(ADAPTER),
    model=tuned_model, tokenizer=tokenizer)

for label, adapter in (("BASE", base_adapter), ("TUNED", tuned_adapter)):
    resp = adapter.generate(rendered.messages, system=rendered.system,
                            params=EVAL_PARAMS)
    print("=" * 78)
    print(f"{label}  error={resp.error}")
    print(f"usage={resp.usage}")
    print(f"diagnostics={ {k: v for k, v in (resp.raw or {}).items() if k != 'decoded_with_special_tokens'} }")
    print(f"\n{resp.text[:900]}\n")
    assert resp.text.strip(), f"{label} returned empty - stop and diagnose above."

print("Both models produced real text.")

## 6. Step 10 — Three held-out scenarios, no judge

One normal, one adversarial, one solved/almost-correct. Six generations, six
non-empty outputs, plus the deterministic checks — before spending a cent on
judging.

In [ ]:
from evaluation.behavioral_checks import run_deterministic_checks

def pick(pred):
    return next((s for s in scenarios if pred(s)), None)

chosen = [
    pick(lambda s: s.pressure_type.value == "normal"),
    pick(lambda s: s.pressure_type.value in
         ("prompt_injection", "authority_override", "repeated_answer_request")),
    pick(lambda s: s.student_has_solved or s.pressure_type.value == "almost_correct"),
]
chosen = [s for s in chosen if s is not None]
print(f"selected: {[(s.id, s.pressure_type.value) for s in chosen]}\n")

attempted = non_empty = 0
for scen in chosen:
    r = strategy.render(scen)
    for label, adapter in (("BASE", base_adapter), ("TUNED", tuned_adapter)):
        resp = adapter.generate(r.messages, system=r.system, params=EVAL_PARAMS)
        attempted += 1
        non_empty += bool(resp.text.strip())
        det = run_deterministic_checks(scen, resp.text, spec)
        print("=" * 78)
        print(f"[{label}] {scen.id} ({scen.pressure_type.value}, "
              f"solved={scen.student_has_solved})")
        print(f"tokens={(resp.usage or {}).get('output_tokens')} "
              f"error={resp.error} violations={det.violations}")
        print(f"\n{resp.text[:600]}\n")

print("=" * 78)
print(f"attempted={attempted}  non_empty={non_empty}")
assert attempted == 6, f"expected 6 generations, got {attempted}"
assert non_empty == 6, "some generations were empty - do NOT run the full rerun yet"
print("SMOKE PASSED.")

## 7. Step 15 — Full rerun into a **new** directory

Only run this once the smoke above passed. Same held-out set, same weak prompt,
same generation settings, same judge, same base revision, same N=600 adapter.
The invalid run is preserved untouched.

In [ ]:
# Free the diagnostic copies first - base_vs_tuned loads its own.
import gc
del base_adapter, tuned_adapter, tuned_model, base_model
gc.collect(); torch.cuda.empty_cache()
print(f"free VRAM: {torch.cuda.mem_get_info()[0]/2**30:.2f} GiB")

import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
!python -m ablations.base_vs_tuned \
    --base "hf:{BASE_MODEL}@{BASE_REVISION}" \
    --tuned "peft:{BASE_MODEL}+outputs/{RUN}" \
    --judge anthropic:claude-opus-5 \
    --eval-set scenarios/heldout.jsonl \
    --strategy zero_shot \
    --max-workers 1 \
    --output results/base_vs_tuned_run2

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path("results/base_vs_tuned_run2/report.md").read_text()))

In [ ]:
# Integrity check on the rerun BEFORE reading any headline number.
import json
res = json.loads(Path("results/base_vs_tuned_run2/results.json").read_text())
for cell in res["cells"]:
    print(f"{cell['label']:6} attempted={cell['attempted_count']} "
          f"measured={cell['scenario_count']} "
          f"infra_errors={cell['infrastructure_error_count']} "
          f"subject_ok={cell['successful_subject_calls']} "
          f"partial={cell['partial']} "
          f"empty={cell['failure_modes'].get('EMPTY_RESPONSE', 0)}")

empties = sum(c["failure_modes"].get("EMPTY_RESPONSE", 0) for c in res["cells"])
if empties:
    print(f"\n!!! {empties} EMPTY_RESPONSE remain - this run is still not "
          f"measuring behavior. Do not interpret the metrics.")
else:
    print("\nNo empty responses. The comparison is measuring behavior.")

In [ ]:
!tar -czf base_vs_tuned_run2.tar.gz results/base_vs_tuned_run2 results/training
from google.colab import files
files.download("base_vs_tuned_run2.tar.gz")

## If the smoke test still fails

Work down this list; each is a different root cause.

| Evidence | Root cause | Action |
| --- | --- | --- |
| `generated_length == 0` | model terminated instantly | Check whether the first token is EOS, and whether the prompt ends mid-turn |
| `generated_length > 0` but text empty | decode/cleanup defect | Compare `skip_special_tokens` True vs False in section 3 |
| `OutOfMemoryError` in the traceback | VRAM exhausted | Confirms the fp32 hypothesis; check the dtype printed in section 1 is `float16` |
| LoRA tensors all zero | adapter never trained | Check the training loss curve; retraining then IS justified |
| Base non-empty, tuned empty | the adapter broke generation | Compare `adapter_config.json` target modules against the training config |
| Both empty even at fp16 | prompt/template problem | Re-read the rendered template in section 2 |

Do not retrain, build Dataset V2, or run the sweep until the smoke passes.